In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from simulation_function import *

### 实验4：从0°扫描到90°，每度做两组模拟
角度范围：0, 1, 2, ..., 90°（共91组×2条线=182次模拟）

每组两条线:
- **baseline**: theta1 = theta2 = angle（两摆在同一直线起始）
- **micro**: theta1 = angle, theta2 = angle + 0.01（差0.01度）

算法：RK4，时长300s

In [2]:
import tqdm
import csv
from pathlib import Path
import numpy as np

In [3]:
# ===== 基础配置 =====
run_number = "01"
folder_path = Path(f"run{run_number}")
folder_path.mkdir(exist_ok=True)

param = {
    "m1": 1.0,
    "m2": 1.0,
    "l1": 1.0,
    "l2": 1.0,
    "g": 9.81,
    "dt": 0.01,
    "duration": 300.0,
    "angle_mode": "DEG",
    "theta1_0": 0.0,
    "theta2_0": 0.0,
    "w1_0": 0.0,
    "w2_0": 0.0,
}

# 角度组：0°到90°（整数度）
angle_groups = list(range(0, 91))

# 微扰量：0.01 度
epsilon = 0.01

duration_str = f"{int(param['duration'])}s"

# 总实验数 = 91组 × 2条线 = 182
total_experiments = len(angle_groups) * 2

# ===== 开始循环 =====
num_counter = 1

with tqdm.tqdm(total=total_experiments, desc="Scanning 0°~90° (RK4)") as pbar:
    for angle in angle_groups:
        # ---- 第1条线：基线 (baseline) ----
        param["theta1_0"] = float(angle)
        param["theta2_0"] = float(angle)
        
        num_str = f"{num_counter:03d}"
        filename = f"run_{run_number}_angle_{angle:03d}deg_{num_str}_baseline_RK4_{param['dt']:.4f}_{duration_str}.csv"
        path = folder_path / filename
        
        temp_storage = simulate_double_pendulum(param, energy=True)
        
        with open(path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['theta_1', 'theta_2', 'energy'])
            writer.writerows(temp_storage)
        
        num_counter += 1
        pbar.update(1)
        
        # ---- 第2条线：微扰 (micro) ----
        param["theta1_0"] = float(angle)
        param["theta2_0"] = float(angle) + epsilon
        
        num_str = f"{num_counter:03d}"
        filename = f"run_{run_number}_angle_{angle:03d}deg_{num_str}_micro_RK4_{param['dt']:.4f}_{duration_str}.csv"
        path = folder_path / filename
        
        temp_storage = simulate_double_pendulum(param, energy=True)
        
        with open(path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['theta_1', 'theta_2', 'energy'])
            writer.writerows(temp_storage)
        
        num_counter += 1
        pbar.update(1)

print("\n🎉 0°~90° 扫描实验数据已完美落地！")
print(f"📁 数据已保存至: {folder_path}")
print(f"📊 共 {len(angle_groups)} 组角度 × 2 条线 = {total_experiments} 次模拟")

Scanning 0°~90° (RK4): 100%|██████████| 182/182 [06:54<00:00,  2.28s/it]


🎉 0°~90° 扫描实验数据已完美落地！
📁 数据已保存至: run01
📊 共 91 组角度 × 2 条线 = 182 次模拟
